In [4]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Создаем Spark Session с PostgreSQL драйвером
spark = SparkSession.builder \
    .appName("pg_spark_fubarbert") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.5.0") \
    .getOrCreate()

print("✅ Spark Session создана!")
print("Spark UI:", spark.sparkContext.uiWebUrl)

✅ Spark Session создана!
Spark UI: http://b7607851a41a:4040


In [5]:
PG_HOST = "postgresql.bootcamp.local" 
PG_PORT = "15432"
PG_DATABASE = "source"
PG_USER = "postgres"
PG_PASSWORD = "4r5t6y7u"

jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
print(f"JDBC URL: {jdbc_url}")

JDBC URL: jdbc:postgresql://postgresql.bootcamp.local:15432/source


In [6]:
shops_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", PG_USER) \
    .option("password", PG_PASSWORD) \
    .option("dbtable", "public.shops") \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print(f"✅ Таблица shops загружена: {shops_df.count()} строк")
shops_df.show(10)

✅ Таблица shops загружена: 10 строк


+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [7]:
shop_timezone_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", PG_USER) \
    .option("password", PG_PASSWORD) \
    .option("dbtable", "public.shop_timezone") \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print(f"✅ Таблица shop_timezone загружена: {shop_timezone_df.count()} строк")
shop_timezone_df.show(10)

✅ Таблица shop_timezone загружена: 15 строк


+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
+-----+---------+
only showing top 10 rows



In [8]:
# Регистрируем временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_timezone_df.createOrReplaceTempView("shop_timezone")

# Выполняем трансформацию
result_df = spark.sql("""
    WITH sz AS (
        SELECT 
            CAST(plant AS INT) AS id, 
            time_zone 
        FROM shop_timezone 
        WHERE CAST(plant AS INT) IS NOT NULL
    ),
    s AS (
        SELECT 
            CAST(st_id AS INT) AS st_id, 
            shop_name 
        FROM shops
    ),
    cte AS (
        SELECT 
            *, 
            ROW_NUMBER() OVER(PARTITION BY st_id ORDER BY st_id) AS rnk 
        FROM s 
        JOIN sz ON s.st_id = sz.id
    )
    SELECT 
        st_id,
        shop_name,
        CAST(
            CASE 
                WHEN time_zone = '' OR time_zone IS NULL THEN 3
                ELSE SUBSTR(time_zone, 4)
            END AS INT
        ) AS tz_code
    FROM cte 
    WHERE rnk = 1
""")

print(f"✅ Трансформация выполнена: {result_df.count()} строк")
result_df.orderBy("st_id").show(100)

✅ Трансформация выполнена: 6 строк


+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      3|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [ ]:
spark.stop()